# Kasafi Bot – Colab backup
Use this if the laptop loses power or internet. **Runtime → Change runtime type → L4 GPU** (Colab Pro).

1. Upload `kasafi-bot.zip` to your Google Drive (My Drive).
2. Put your secrets in Colab **Secrets** (key icon on the left): `HF_TOKEN`, `TELEGRAM_BOT_TOKEN`, `LOG_SALT`.
3. Run all cells. The Telegram bot runs here until you stop it (keep the tab open).

Stop the laptop bot first: Telegram allows only one running copy per bot token.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
!unzip -oq /content/drive/MyDrive/kasafi-bot.zip -d /content/
%cd /content/kasafi-bot
!apt-get -qq install -y ffmpeg > /dev/null
!pip -q install -r requirements.txt

In [ ]:
import os
env = {
  'HF_TOKEN': userdata.get('HF_TOKEN'),
  'TELEGRAM_BOT_TOKEN': userdata.get('TELEGRAM_BOT_TOKEN'),
  'LOG_SALT': userdata.get('LOG_SALT'),
  'LLM_BACKEND': 'transformers', 'LLM_4BIT': '0', 'ASR_ENABLED': '1',
  # keep the interaction log on Drive so evidence survives the session
  'LOG_DB': '/content/drive/MyDrive/kasafi_interactions.sqlite',
}
with open('.env', 'w') as f:
    f.write('\n'.join(f'{k}={v}' for k, v in env.items()))
!python -m tests.test_questions | tail -1
!python scripts/check_models.py

In [ ]:
# Runs Telegram (polling). Interrupt this cell to stop.
!python -m app.run_all --no-whatsapp

**Merging logs:** if both the laptop and Colab collected interactions, copy `kasafi_interactions.sqlite` from Drive to the laptop and run `python scripts/merge_logs.py <that file>` before exporting evidence.